This script ingests data through API from CMS datasets for The Medicare Physician & Other Practitioners by Provider and Service dataset provides information on use, payments, and submitted charges organized by National Provider Identifier (NPI), Healthcare Common Procedure Coding System (HCPCS) code, and place of service.

The last update of the page was on May 21, 2026 when the was available through the year 2024.

Import packages, call the existing table to see if the specialty already exists in the table; if not, import the specialty list. 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from api_func import fetch_cms

import json, re, os, time

dbutils.widgets.text("specialty", "")


specialty = dbutils.widgets.get("specialty").strip()


spark = SparkSession.builder.getOrCreate()


# the table has been created
table_name = "northstar_network_bronze.cms_provider_specialty.api_ingestion_table"
provider_table = spark.table(table_name)

exists = provider_table.filter(F.col("Rndrng_Prvdr_Type") == specialty).limit(1).count() > 0

if not exists:
    cms_data = fetch_cms(provider_type=specialty)

    try:
        cms_df = spark.createDataFrame(cms_data)
        cms_df = cms_df.withColumn("_ingest_date", F.current_date()).withColumn("_ingest_time", F.current_timestamp())
        cms_df.write.mode("append").saveAsTable(table_name)
        print(f"Added {specialty} to the table.")

    except Exception as e:
        print(f"Error: {e}")
        print("Check speciallty name in cms data - https://data.cms.gov/")

else:
    print(f"Specialty already exists. Skipping {specialty}.")


